In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor

In [1]:


riders = pd.read_csv("../dataset/processed/riders_features.csv")

riders.head()

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,...,Day_of_Week,Month,Weekend,Peak_Period,Trip_Distance_km,Traffic_Score,Weather_Score,Vehicle_Score,Rider_Experience,Workload
0,0xcdcd,DEHRES17DEL01,36.0,4.2,30.327968,78.046106,30.397968,78.116106,2022-02-12,21:55,...,Saturday,2,1,Dinner,10.280582,4,4,4,151.2,3.0
1,0xd987,KOCRES16DEL01,21.0,4.7,10.003064,76.307589,10.043064,76.347589,2022-02-13,14:55,...,Sunday,2,1,Lunch,6.242319,3,5,4,98.7,1.0
2,0x2784,PUNERES13DEL03,23.0,4.7,18.562450,73.916619,18.652450,74.006619,2022-03-04,17:30,...,Friday,3,0,Normal,13.787860,2,6,3,108.1,1.0
3,0xc8b6,LUDHRES15DEL02,34.0,4.3,30.899584,75.809346,30.919584,75.829346,2022-02-13,09:20,...,Sunday,2,1,Breakfast,2.930258,1,6,4,146.2,0.0
4,0xdb64,KNPRES14DEL02,24.0,4.7,26.463504,80.372929,26.593504,80.502929,2022-02-14,19:50,...,Monday,2,0,Dinner,19.396618,4,4,3,112.8,1.0


O2A(Order to Assignment)
 

In [2]:
riders["O2A_Target"] = (
    0.05 * riders["Time_taken (min)"] +
    0.5 * riders["multiple_deliveries"] +
    0.3 * riders["Traffic_Score"]
)

FM(First Mile)

In [3]:
riders["FM_Target"] = (
    0.20 * riders["Time_taken (min)"] +
    0.25 * riders["Trip_Distance_km"]
)

WT (Wait Time)

In [4]:
riders["WT_Target"] = (
    0.25 * riders["Time_taken (min)"] +
    0.5 * riders["Weather_Score"]
)

LM(Last mile)

In [5]:
riders["LM_Target"] = (
    riders["Time_taken (min)"]
    - riders["O2A_Target"]
    - riders["FM_Target"]
    - riders["WT_Target"]
)

In [6]:
print(riders[[
    "Time_taken (min)",
    "O2A_Target",
    "FM_Target",
    "WT_Target",
    "LM_Target"
]].head())

   Time_taken (min)  O2A_Target  FM_Target  WT_Target  LM_Target
0                46        5.00  11.770146      13.50  15.729854
1                23        2.55   6.160580       8.25   6.039420
2                21        2.15   7.646965       8.25   2.953035
3                20        1.30   4.732564       8.00   5.967436
4                41        3.75  13.049155      12.25  11.950845


In [7]:
eta = (
    riders["O2A_Target"] +
    riders["FM_Target"] +
    riders["WT_Target"] +
    riders["LM_Target"]
)

print((eta - riders["Time_taken (min)"]).abs().max())

7.105427357601002e-15


In [8]:
drop_columns = [
    "ID",
    "Order_Date",
    "Time_Orderd",
    "Time_Order_picked",
    "Order_Time",
    "Time_taken (min)",
    "O2A_Target",
    "FM_Target",
    "WT_Target",
    "LM_Target"
]

X = riders.drop(columns=drop_columns)

In [10]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

numerical_features = X.select_dtypes(exclude=["object"]).columns.tolist()

In [11]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [12]:
def train_stage_model(target_column):

    y = riders[target_column]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model",
             XGBRegressor(
                 n_estimators=300,
                 learning_rate=0.05,
                 max_depth=8,
                 subsample=0.8,
                 colsample_bytree=0.8,
                 objective="reg:squarederror",
                 random_state=42
             ))
        ]
    )

    model.fit(X_train, y_train)

    prediction = model.predict(X_test)

    mae = mean_absolute_error(y_test, prediction)
    rmse = np.sqrt(mean_squared_error(y_test, prediction))
    r2 = r2_score(y_test, prediction)

    print("="*60)
    print(target_column)
    print("="*60)
    print(f"MAE  : {mae:.3f}")
    print(f"RMSE : {rmse:.3f}")
    print(f"R²   : {r2:.4f}")

    return model

In [13]:
o2a_model = train_stage_model("O2A_Target")

O2A_Target
MAE  : 0.154
RMSE : 0.193
R²   : 0.9483


In [14]:
wt_model = train_stage_model("WT_Target")

WT_Target
MAE  : 0.764
RMSE : 0.961
R²   : 0.8592


In [15]:
lm_model = train_stage_model("LM_Target")

LM_Target
MAE  : 1.552
RMSE : 1.955
R²   : 0.7861


In [16]:
fm_model = train_stage_model("FM_Target")

FM_Target
MAE  : 0.620
RMSE : 0.780
R²   : 0.9160


In [18]:
import joblib

joblib.dump(o2a_model, "../models/o2a_model.pkl")
joblib.dump(fm_model, "../models/fm_model.pkl")
joblib.dump(wt_model, "../models/wt_model.pkl")
joblib.dump(lm_model, "../models/lm_model.pkl")

['../models/lm_model.pkl']